# Deep-Bit Flip — perturb only the deepest 10% of the tree encoding

**The professor's idea:** the shallow tree questions are learned from thousands of rows (real knowledge — don't touch them); the **deepest** questions are learned from a handful of rows (rote memorization). So flip/perturb **only the deepest ~10% of bits** — it shouldn't be damaging, and it attacks memorization exactly where it lives. This is the *hard* (on/off) form of depth-dependent noise.

**Implementation:** bits stay **binary** (no sigmoid, no τ — the confound that sank the last experiment can't happen). Each targeted bit flips 0↔1 with probability p, fresh on every training batch; eval always uses clean bits. Targeted bits = deepest by depth, ties broken by fewest training samples.

### The four arms (credit dataset, x+tree, OOB-honest, all else identical)
| arm | what changes | question |
|---|---|---|
| control | nothing | the 0.8473 baseline |
| flip p=0.25 | deep 10% flip 25% of the time | gentle scramble |
| flip p=0.5 | deep 10% become pure coin flips | total scramble |
| **delete** | deep 10% zeroed everywhere | **the diagnostic**: if deep bits are pure memorization, deleting them should barely hurt |

**Reading the outcome:** flip ≈/> control with train-AUC off 1.0 → professor confirmed; delete barely hurts → deep bits carry no real signal (a finding by itself); flips clearly worse → deep bits do carry signal on this dataset.

⏱ ~20–30 min on an A100. Runtime → GPU → Run all.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 3b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import os, glob, tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 4 · THE FOUR ARMS  (hard bits everywhere — no tau)
base = ('--task 361055 --views x+tree --encoding oob --ensemble 2 --epochs 400 '
        '--dropout 0 --l1 0 --weight-decay 1e-3 --lr 3e-4 --batch-size 128 --device auto')
!python -u run_fusion.py {base}                                        --out results/fusion/df_control
!python -u run_fusion.py {base} --deep-frac 0.1 --deep-flip-p 0.25     --out results/fusion/df_flip25
!python -u run_fusion.py {base} --deep-frac 0.1 --deep-flip-p 0.5      --out results/fusion/df_flip50
!python -u run_fusion.py {base} --deep-frac 0.1 --deep-delete          --out results/fusion/df_delete
# optional extra: delete a wider slice to find where signal starts to die
# !python -u run_fusion.py {base} --deep-frac 0.25 --deep-delete       --out results/fusion/df_delete25

In [ ]:
# 5 · summary table
import json, glob, pandas as pd
rows = []
for d in sorted(glob.glob('results/fusion/df_*')):
    s = json.load(open(glob.glob(d + '/fusion_*.json')[0]))
    e = pd.read_csv(glob.glob(d + '/fusion_*_epochs.csv')[0])
    r = s['results'][0]
    rows.append(dict(arm=d.split('df_')[-1], flip_p=s.get('deep_flip_p', 0),
                     delete=s.get('deep_delete', False),
                     test_auc=r['test_auc'], best_val=r['best_val_auc'],
                     final_train_auc=e.groupby('epoch').train_auc.mean().iloc[-1],
                     ceiling=s['tree_ceiling']))
t = pd.DataFrame(rows)
print(t.round(4).to_string(index=False))
c = t[t.arm=='control'].test_auc.iloc[0]
print(f'\ncontrol {c:.4f} | ceiling {t.ceiling.iloc[0]:.4f} | noise band ~ +/-0.01')
for _, r in t[t.arm!='control'].iterrows():
    print(f"  {r.arm:10s} vs control: {r.test_auc - c:+.4f}")

In [ ]:
# 6 · show every figure
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/fusion/df_*/*.png')):
    print('==', f, '==')
    display(Image(f))

In [ ]:
# 7 · download everything
import shutil
from google.colab import files
shutil.make_archive('deepflip_credit', 'zip', 'results/fusion')
files.download('deepflip_credit.zip')